# Inference: Bootstrap Confidence Intervals

Estimate confidence intervals for each coefficient using non-parametric bootstrap and compare to the standard regression CIs.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
np.random.seed(42)


In [ ]:

# Load preprocessed training data and create encoded feature matrix
def load_encoded_train(train_path="data/train_data_preprocessed.csv"):
    df = pd.read_csv(train_path)
    X = df.drop(columns=["y"])
    y = df["y"].map({"yes": 1, "no": 0})
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    return X_enc, y

# Encode holdout with train columns to keep alignment
# If you already have a preprocessed holdout file, point holdout_path there instead.
def load_encoded_holdout(train_cols, holdout_path="data/holdout_data.csv"):
    holdout_df = pd.read_csv(holdout_path)
    if "y" in holdout_df.columns:
        y_holdout = holdout_df["y"].map({"yes": 1, "no": 0})
        X_holdout = holdout_df.drop(columns=["y"])
    else:
        y_holdout = None
        X_holdout = holdout_df
    cat_cols = [c for c in X_holdout.columns if X_holdout[c].dtype == "object"]
    X_holdout_enc = pd.get_dummies(X_holdout, columns=cat_cols, drop_first=True)
    # align to train columns
    X_holdout_enc = X_holdout_enc.reindex(columns=train_cols, fill_value=0)
    return X_holdout_enc, y_holdout

train_X, train_y = load_encoded_train()
holdout_X, holdout_y = load_encoded_holdout(train_X.columns)
print(f"Train encoded shape: {train_X.shape}; Holdout encoded shape: {holdout_X.shape}")


In [ ]:

alpha = 0.05
n_boot = 200  # increase if you need tighter intervals

X_const = sm.add_constant(train_X)
logit_train = sm.Logit(train_y, X_const)
train_res = logit_train.fit(disp=False)
standard_ci = train_res.conf_int(alpha=alpha)
standard_ci.columns = ['ci_lower_standard', 'ci_upper_standard']

coef_names = standard_ci.index
boot_coefs = {name: [] for name in coef_names}

for i in range(n_boot):
    sample_idx = np.random.choice(len(train_y), size=len(train_y), replace=True)
    X_sample = X_const.iloc[sample_idx]
    y_sample = train_y.iloc[sample_idx]
    try:
        res = sm.Logit(y_sample, X_sample).fit(disp=False)
        for name in coef_names:
            boot_coefs[name].append(res.params[name])
    except Exception:
        # skip failed fits
        continue

boot_ci = {}
for name in coef_names:
    vals = np.array(boot_coefs[name])
    if len(vals) == 0:
        boot_ci[name] = (np.nan, np.nan)
        continue
    lower = np.percentile(vals, 100 * alpha / 2)
    upper = np.percentile(vals, 100 * (1 - alpha / 2))
    boot_ci[name] = (lower, upper)

ci_df = standard_ci.copy()
ci_df['ci_lower_boot'] = [boot_ci[name][0] for name in coef_names]
ci_df['ci_upper_boot'] = [boot_ci[name][1] for name in coef_names]
ci_df['boot_samples_used'] = [len(boot_coefs[name]) for name in coef_names]
display(ci_df.head(15))


### Discussion
Compare bootstrap vs. standard regression CIs. Note any widening/narrowing and explain which interval type you would report to a stakeholder and why (e.g., robustness to non-normality, small-sample corrections).